In [1]:
spark

NameError: name 'spark' is not defined

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("ADFTransformations").getOrCreate()

In [3]:
spark

In [30]:
from pyspark.sql.functions import col, sum, avg, when, expr, rank

# 1. DataFrame Creation

## Employee Data

In [ ]:
# emp_filePath = 'dbfs:/FileStore/shared_uploads/iriscloudone@outlook.com/emp.csv' -- Azure Databricks

In [ ]:
# dep_filePath = 'dbfs:/FileStore/shared_uploads/iriscloudone@outlook.com/dep.csv' -- Azure Databricks

In [4]:
#employee_df = spark.read.csv(emp_filePath,header=True,inferSchema=True) -- Azure Databricks
employee_df = spark.read.csv('data/emp.csv',header=True,inferSchema=True)

In [5]:
#department_df = spark.read.csv(dep_filePath,header=True,inferSchema=True) -- Azure Databricks
department_df = spark.read.csv('data/dep.csv',header=True,inferSchema=True)

In [6]:
employee_df.show()

+------+---------------+-------------+------+
|emp_id|       emp_name|department_id|salary|
+------+---------------+-------------+------+
|     1|       John Doe|          101| 50000|
|     2|     Jane Smith|          102| 60000|
|     3|Michael Johnson|          101| 55000|
|     4|    Emily Davis|          103| 48000|
|     5|   Robert Brown|          102| 62000|
|     6|    Emma Wilson|          101| 51000|
|     7|  William Jones|          103| 49000|
|     8|Olivia Martinez|          102| 63000|
|     9|    Liam Taylor|          101| 52000|
|    10|Sophia Anderson|          103| 50000|
+------+---------------+-------------+------+



In [7]:
department_df.show()

+-------------+---------------+-------------+
|department_id|department_name|     location|
+-------------+---------------+-------------+
|          101|    Engineering|     New York|
|          102|      Marketing|San Francisco|
|          103|        Finance|      Chicago|
|          104|Human Resources|  Los Angeles|
+-------------+---------------+-------------+



In [8]:
# 2. Select Pick specific columns.
selected_df = employee_df.select("emp_name", "salary")

In [9]:
selected_df.show()

+---------------+------+
|       emp_name|salary|
+---------------+------+
|       John Doe| 50000|
|     Jane Smith| 60000|
|Michael Johnson| 55000|
|    Emily Davis| 48000|
|   Robert Brown| 62000|
|    Emma Wilson| 51000|
|  William Jones| 49000|
|Olivia Martinez| 63000|
|    Liam Taylor| 52000|
|Sophia Anderson| 50000|
+---------------+------+



In [10]:
employee_df['emp_name','salary'].show()

+---------------+------+
|       emp_name|salary|
+---------------+------+
|       John Doe| 50000|
|     Jane Smith| 60000|
|Michael Johnson| 55000|
|    Emily Davis| 48000|
|   Robert Brown| 62000|
|    Emma Wilson| 51000|
|  William Jones| 49000|
|Olivia Martinez| 63000|
|    Liam Taylor| 52000|
|Sophia Anderson| 50000|
+---------------+------+



In [ ]:
# 3. Filter - Filter employees over salary 5000.
filtered_df = employee_df.filter(col("salary") > 50000)

In [ ]:
filtered_df.show()

In [12]:
employee_df.filter(employee_df['salary']>50000).show()

+------+---------------+-------------+------+
|emp_id|       emp_name|department_id|salary|
+------+---------------+-------------+------+
|     2|     Jane Smith|          102| 60000|
|     3|Michael Johnson|          101| 55000|
|     5|   Robert Brown|          102| 62000|
|     6|    Emma Wilson|          101| 51000|
|     8|Olivia Martinez|          102| 63000|
|     9|    Liam Taylor|          101| 52000|
+------+---------------+-------------+------+



In [ ]:
# 4. Join - Join Employee with Department.
joined_df = employee_df.join(department_df, on="department_id", how="inner")

In [ ]:
joined_df.show()

In [ ]:
display(joined_df)

In [ ]:
joined_df.show()

In [ ]:
# 5. Aggregate - Sum of salaries per department.
aggregated_df = employee_df.groupBy("department_id").agg(sum("salary").alias("Totalsalary"))

In [ ]:
aggregated_df.show()

In [ ]:
# 6. Aggregate - Avg of salaries per department.
aggregated_df = employee_df.groupBy("department_id").agg(avg("salary").alias("AvgSalarybyDep"))

In [ ]:
aggregated_df.show()

In [ ]:
# 7. Derived Column - Add 10% increased salary.
derived_df = employee_df.withColumn("salaryIncrease", col("salary") * 1.1)

In [ ]:
derived_df.show()

In [ ]:
# 9. Sort - Sort employees by descending salary.
sorted_df = employee_df.orderBy(col("salary").desc())

In [ ]:
sorted_df.show()

In [ ]:
aggregated_df = employee_df.groupBy("department_id").agg(avg("salary").alias("AvgSalarybyDep")).orderBy("department_id")

In [ ]:
aggregated_df.show()

In [13]:
# 10. Lookup (Join) add department names to employee data.
lookup_df = employee_df.join(department_df, on="department_id", how="left")

In [14]:
lookup_df.show()

+-------------+------+---------------+------+---------------+-------------+
|department_id|emp_id|       emp_name|salary|department_name|     location|
+-------------+------+---------------+------+---------------+-------------+
|          101|     1|       John Doe| 50000|    Engineering|     New York|
|          102|     2|     Jane Smith| 60000|      Marketing|San Francisco|
|          101|     3|Michael Johnson| 55000|    Engineering|     New York|
|          103|     4|    Emily Davis| 48000|        Finance|      Chicago|
|          102|     5|   Robert Brown| 62000|      Marketing|San Francisco|
|          101|     6|    Emma Wilson| 51000|    Engineering|     New York|
|          103|     7|  William Jones| 49000|        Finance|      Chicago|
|          102|     8|Olivia Martinez| 63000|      Marketing|San Francisco|
|          101|     9|    Liam Taylor| 52000|    Engineering|     New York|
|          103|    10|Sophia Anderson| 50000|        Finance|      Chicago|
+-----------

In [15]:
# 11. # Pivot the data by 'department_id' and aggregate 'salary'
from pyspark.sql.functions import sum
pivot_df = employee_df.groupBy().pivot("department_id").agg(sum("salary"))

In [16]:
pivot_df.show()

+------+------+------+
|   101|   102|   103|
+------+------+------+
|208000|185000|147000|
+------+------+------+



In [ ]:
# 13. Copy Activity - Simulate copy by writing to file/database (e.g., parquet).
# employee_df.write.mode("overwrite").parquet("data/employee.parquet") --- Azure Databricks

In [18]:
# 14. Union -Combine employees in HR and IT.
hr_df = employee_df.filter(col("department_id") == 101)

In [ ]:
it_df = employee_df.filter(col("department_id") == 102)

In [ ]:
union_df = hr_df.union(it_df)

In [19]:
hr_df.show()

+------+---------------+-------------+------+
|emp_id|       emp_name|department_id|salary|
+------+---------------+-------------+------+
|     1|       John Doe|          101| 50000|
|     3|Michael Johnson|          101| 55000|
|     6|    Emma Wilson|          101| 51000|
|     9|    Liam Taylor|          101| 52000|
+------+---------------+-------------+------+



In [20]:
it_df = employee_df.filter(col("department_id") == 102)

In [21]:
it_df.show()

+------+---------------+-------------+------+
|emp_id|       emp_name|department_id|salary|
+------+---------------+-------------+------+
|     2|     Jane Smith|          102| 60000|
|     5|   Robert Brown|          102| 62000|
|     8|Olivia Martinez|          102| 63000|
+------+---------------+-------------+------+



In [22]:
union_df = hr_df.union(it_df)

In [23]:
union_df.show()

+------+---------------+-------------+------+
|emp_id|       emp_name|department_id|salary|
+------+---------------+-------------+------+
|     1|       John Doe|          101| 50000|
|     3|Michael Johnson|          101| 55000|
|     6|    Emma Wilson|          101| 51000|
|     9|    Liam Taylor|          101| 52000|
|     2|     Jane Smith|          102| 60000|
|     5|   Robert Brown|          102| 62000|
|     8|Olivia Martinez|          102| 63000|
+------+---------------+-------------+------+



In [34]:
altered_df = employee_df.withColumn(
    "Operation",
    when(col("salary") < 60000, "LessthanBaseSalary")
    .when(col("salary") >= 60000, "GreatherthanBaseSalary")
)

In [35]:
altered_df.show()

+------+---------------+-------------+------+--------------------+
|emp_id|       emp_name|department_id|salary|           Operation|
+------+---------------+-------------+------+--------------------+
|     1|       John Doe|          101| 50000|  LessthanBaseSalary|
|     2|     Jane Smith|          102| 60000|GreatherthanBaseS...|
|     3|Michael Johnson|          101| 55000|  LessthanBaseSalary|
|     4|    Emily Davis|          103| 48000|  LessthanBaseSalary|
|     5|   Robert Brown|          102| 62000|GreatherthanBaseS...|
|     6|    Emma Wilson|          101| 51000|  LessthanBaseSalary|
|     7|  William Jones|          103| 49000|  LessthanBaseSalary|
|     8|Olivia Martinez|          102| 63000|GreatherthanBaseS...|
|     9|    Liam Taylor|          101| 52000|  LessthanBaseSalary|
|    10|Sophia Anderson|          103| 50000|  LessthanBaseSalary|
+------+---------------+-------------+------+--------------------+



In [ ]:
employee_df.write \
    .format("jdbc") \
    .option("url", "jdbc:sqlserver://<server>:1433;databaseName=<db>") \
    .option("dbtable", "Employee") \
    .option("user", "<user>") \
    .option("password", "<password>") \
    .save()